# 4. LLM-режим команды Graph-EVT-agent

**Цель:** запустить детерминированный численный `GraphEVTPipeline`, а затем передать его ограниченный evidence-пакет специализированным LLM-агентам из Graph-EVT-agent. LLM объясняет и проверяет результат, но не меняет порог, граф или ranking.

Для живого запуска задайте `MISTRAL_API_KEY` в окружении или введите его в скрытом prompt ячейки ниже. Ключ не записывается в ноутбук или артефакты.

In [1]:
from pathlib import Path
import os
from getpass import getpass
import numpy as np
import pandas as pd
from graph_evt_agent import EVTConfig, GraphConfig, GraphEVTPipeline, InputConfig
from graph_evt_agent.agents import EVTAgentTeam, MistralAPIError, MistralClient

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/generated/fractal_extreme_series.csv").exists())
DATA = ROOT / "data/generated/fractal_extreme_series.csv"
SEED = 42
df = pd.read_csv(DATA)
x = df["value"].to_numpy()
assert np.isfinite(x).all() and x.ndim == 1


In [5]:
api_key = os.getenv("MISTRAL_API_KEY") or getpass("Mistral API key (input hidden): ").strip()
MISTRAL_READY = False
mistral_client = None

if api_key:
    try:
        mistral_client = MistralClient(
            api_key=api_key,
            model="ministral-3b-2512",
            max_retries=0,
        )
        check_reply = mistral_client.complete(
            "Reply exactly with MISTRAL_OK.",
            "Connection check. Reply exactly with MISTRAL_OK.",
        )
        MISTRAL_READY = bool(check_reply.strip())
        print("Mistral API check passed." if MISTRAL_READY else "Mistral returned an empty response.")
    except MistralAPIError as error:
        error_text = str(error).lower()
        if "429" in error_text or "rate_limit" in error_text or "rate limit" in error_text:
            print("Mistral rate limit (HTTP 429). Check your account usage and rate limits, wait, then rerun this cell.")
        else:
            print(f"Mistral API check failed: {error}")
    except Exception as error:
        print(f"Mistral API check failed ({type(error).__name__}).")
else:
    print("Mistral API check skipped: no key was provided.")


Mistral API check passed.


In [6]:
BASELINE_SIZE = int(0.75 * len(x))
pipeline = GraphEVTPipeline(
    EVTConfig(baseline_size=BASELINE_SIZE, tail_quantile=0.95, alarm_probability=0.999, persistence=3),
    graph=GraphConfig(random_state=SEED),
    input_config=InputConfig(missing="error"),
)
# Численный результат доступен и воспроизводим независимо от LLM.
numeric_result = pipeline.run_detailed(x)
{"detected": numeric_result.detection.detected, "time_index": numeric_result.detection.time_index}

{'detected': False, 'time_index': None}

In [7]:
if MISTRAL_READY:
    team = EVTAgentTeam(pipeline, mistral_client)
    try:
        agent_report = team.run(
            x,
            task="Проверь EVT-детекцию и ограничения вывода для одномерного синтетического ряда",
        )
        print(agent_report.final_report)
    except MistralAPIError as error:
        error_text = str(error).lower()
        if "429" in error_text or "rate_limit" in error_text or "rate limit" in error_text:
            print("Mistral rate limit during review. Check account usage/rate limits, wait, then rerun this cell.")
        else:
            print(f"LLM review failed: {error}")
    except Exception as error:
        print(f"LLM review failed ({type(error).__name__}).")
else:
    print("LLM review skipped; the numeric pipeline result is still available.")


### **Краткий итог анализа данных**

#### **1. Общая структура данных**
- **Тип профиля**: Одномерный (`univariate`), `n_channels = 1` (одноканальный).
- **Объём данных**: 1500 точек (`n_time = 1500`), без пропусков (`missing_per_channel = [0]`).

#### **2. Статистическая редкость**
- **Показатели EVT-индикатора**:
  - Минимальное значение: **0.0**
  - Максимальное значение: **~9.4** (пределы варьируются в пределах `[0.0, 9.388]`).
  - **Интерпретация**: Данные соответствуют стабильному синтетическому ряду без значимых аномалий. Порог детекции (`alarm_threshold = 9.2`) близок к минимуму, что может указывать на низкую чувствительность к отклонениям.

#### **3. Ограничения и причинность**
- **Отсутствие временных меток** (`has_timestamps = false`) и интервала (`sampling_interval = null`) делает невозможной:
  - Точную интерпретацию временных событий.
  - Проверку регулярности данных.
- **Потенциальные причины**:
  - Синтетический ряд без аномалий может не срабатывать EVT-детектор.
  - Дл

`EVTAgentTeam` сначала вызывает тот же детерминированный pipeline. Агентам передаются агрегированные evidence-поля, а не право переписать вычисления. Для 1-D входа графовый и локализационный агенты корректно пропускаются. Текст LLM является интерпретацией и должен проверяться по `numeric_result`.